# Beyond the Cluster: Which Israeli Local Authorities Out- and Under-Perform Their Socio-Economic Standing in National Achievement Tests?

**Introduction to Data Science — Final Project**
Stav Tchizik (316062835) · Or Cohen (206631509)
Ministry of Education Meitzav scores, 2016–2018 × CBS Socio-Economic Index of Local Authorities, 2017 edition

---

## Research question

**Which local authorities in Israel score higher or lower on national tests than their socio-economic level
would predict, and what sets those authorities apart?**

Two smaller questions support it: how closely the socio-economic index tracks test scores in the first place,
which gives us the baseline the deviations are measured against, and whether grouping authorities by their
achievement profiles reproduces the official CBS socio-economic clusters (1–10).


## Setup

Imports and one random seed, `RANDOM_STATE = 42`. Every step that involves randomness later on (the train /
validation / test split, cross-validation, Random Forests, K-Means) takes its seed from this constant, so the
results can be reproduced exactly.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

DATA_DIR = Path("datasets")
EDUCATION_PATH = DATA_DIR / "performances_by_areas_years_2015_2019.csv"
CBS_PATH = DATA_DIR / "t01_2017.xls"

## Loading the data

**Ministry of Education Meitzav scores.** A clean CSV with one row per authority × year × sector (`norm_id`) ×
grade (`shichva`) × subject. It loads as is.

**CBS socio-economic index, 2017 edition.** The Excel sheet `t01` was laid out to be printed, not analysed:

- Rows 0–9 are a bilingual (Hebrew / English) header spread over several rows. Rows 8–9 inside that block are
  not authorities but national summary rows (`AVERAGE` and `NATIONWIDE VALUE`).
- The 255 authorities sit in rows 10–264.
- After a blank row, the last 5 rows are footnotes (`*`, `**`, `***`, `(1)`, `..`).

We skip the header block, keep only rows that have a numeric 1–10 cluster (which drops the blank row and the
footnotes), and check by name that neither national summary row got through. Instead of the multi-row header
we give the columns short English names. Each of the 14 variables the index is built from comes as three
columns: the raw value, a standardized value (`_z`) and the rank among the 255 authorities (`_rank`). The CBS
marks suppressed cells with `..` ("not for publication"), and these become missing values.

In [ ]:
education = pd.read_csv(EDUCATION_PATH)

# The 14 variables the CBS index is built from, in the order they appear in the sheet
CBS_VARIABLES = [
    "median_age",
    "dependency_ratio",
    "pct_families_4plus_children",
    "avg_years_schooling",          # ages 25-54
    "pct_academic_degree",          # ages 27-54
    "pct_earners",                  # ages 25-54, income from work
    "pct_women_no_work_income",     # women aged 25-54
    "pct_above_twice_avg_wage",
    "pct_below_min_wage",
    "pct_income_support",           # ages 20+, incl. old-age pension supplement
    "income_per_capita",            # average monthly income per capita
    "vehicles_per_100",             # owned vehicles per 100 residents aged 17+
    "avg_vehicle_licence_fee",
    "avg_days_abroad",
]

CBS_COLUMNS = (
    ["municipal_status", "locality_code", "name_he", "population",
     "ses_index", "ses_rank", "ses_cluster"]
    + [f"{var}{suffix}" for var in CBS_VARIABLES for suffix in ("", "_z", "_rank")]
    + ["name_en"]
)

cbs_raw = pd.read_excel(CBS_PATH, sheet_name="t01", header=None, skiprows=10)
assert cbs_raw.shape[1] == len(CBS_COLUMNS), "unexpected number of columns in t01"
cbs_raw.columns = CBS_COLUMNS

# Real authorities are the only rows with a 1-10 cluster; this drops the blank row and the 5 footnotes
is_authority = pd.to_numeric(cbs_raw["ses_cluster"], errors="coerce").between(1, 10)
cbs = cbs_raw[is_authority].reset_index(drop=True)

# The national AVERAGE / NATIONWIDE VALUE rows sit inside the skipped header block - make sure
national_rows = cbs["name_en"].str.strip().isin(["AVERAGE", "NATIONWIDE VALUE"])
assert not national_rows.any(), "national summary rows were not removed"

# Everything except the two name columns is numeric; '..' (not for publication) becomes NaN
numeric_cols = cbs.columns.difference(["name_he", "name_en"])
cbs[numeric_cols] = cbs[numeric_cols].apply(pd.to_numeric, errors="coerce")
cbs["name_he"] = cbs["name_he"].str.strip()
cbs["name_en"] = cbs["name_en"].str.strip()
for col in ["municipal_status", "ses_rank", "ses_cluster"]:
    cbs[col] = cbs[col].astype("Int64")
cbs["locality_code"] = cbs["locality_code"].astype("Int64")  # regional councils have no locality code

print(f"CBS rows kept: {len(cbs)} of {len(cbs_raw)} read "
      f"({len(cbs_raw) - len(cbs)} blank / footnote rows dropped)")
cbs.head()

### Dataset summary

Rows, columns, years covered and number of distinct authorities in each file (Table 1).

In [ ]:
dataset_summary = pd.DataFrame(
    {
        "Ministry of Education (Meitzav)": {
            "rows": len(education),
            "columns": education.shape[1],
            "grain": "authority × year × sector × grade × subject",
            "years": f"{education['year'].min()}–{education['year'].max()}",
            "authorities": education["rashut"].nunique(),
        },
        "CBS socio-economic index": {
            "rows": len(cbs),
            "columns": cbs.shape[1],
            "grain": "authority",
            "years": "2017 (single edition)",
            "authorities": cbs["name_he"].nunique(),
        },
    }
)
print("Meitzav rows per year:", education["year"].value_counts().sort_index().to_dict())
print("CBS authorities with a locality code:", cbs["locality_code"].notna().sum(),
      "| regional councils (no locality code):", cbs["locality_code"].isna().sum())
dataset_summary